# Initial Project Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted successfully!")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib, time

from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score


pd.set_option("display.width", 120)
print("Libraries loaded.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted successfully!
Libraries loaded.


# Load and clean the raw data

Initially working with this dataset it's cleaned to turn the raw csv data into an actual clean table.

First, the stray index column, the unnamed column, is dropped as it doesn't contain real data. Afterwhich the vitals are forced to become real numbers, instead of text. As a result text such as "120 bpm" becomes NaN. Then the ESI labels must be a number ranging from 1 - 5. The rows in which that data is missing or out of range is unsuited to train a triage model; that data is dropped. Then, physically impossible vital signs are blanked out such that it doesn't mislead the model. The gender column now operates like a flag, using numbers like 0 and 1 to indicate male and female. Other missing numbers in the columns are filled wiuth the column's median number. 

In [2]:
import os 
RAW = "/content/drive/MyDrive/yaleemmlc_admissionprediction_triage.csv"
df_raw = pd.read_csv(RAW)
print("Loaded", df_raw.shape[0], "rows and", df_raw.shape[1], "columns from:", RAW)
df_raw.head()

Loaded 55121 rows and 226 columns from: /content/drive/MyDrive/yaleemmlc_admissionprediction_triage.csv


,Unnamed: 0,dep_name,esi,age,gender,ethnicity,race,lang,religion,maritalstatus,...,cc_vaginaldischarge,cc_vaginalpain,cc_weakness,cc_wheezing,cc_withdrawal-alcohol,cc_woundcheck,cc_woundinfection,cc_woundre-evaluation,cc_wristinjury,cc_wristpain
0,7,A,4.0,87.0,Female,Hispanic or Latino,Other,Other,Pentecostal,Widowed,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,17,B,2.0,53.0,Male,Hispanic or Latino,Other,English,Catholic,Significant Other,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,40,A,2.0,49.0,Female,Non-Hispanic,White or Caucasian,English,Catholic,Married,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,47,A,3.0,22.0,Female,Hispanic or Latino,White or Caucasian,English,Catholic,Single,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,60,A,2.0,62.0,Male,Non-Hispanic,White or Caucasian,English,Protestant,Divorced,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
VITALS = ["triage_vital_hr", "triage_vital_sbp", "triage_vital_dbp", "triage_vital_rr",
          "triage_vital_o2", "triage_vital_temp", "triage_glucose"]
df = df_raw.copy()

df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")], errors="ignore")

for col in VITALS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["esi"] = pd.to_numeric(df["esi"], errors="coerce")
df = df[df["esi"].isin([1, 2, 3, 4, 5])].copy()

df.loc[(df["triage_vital_temp"] < 90) | (df["triage_vital_temp"] > 110), "triage_vital_temp"] = np.nan
df.loc[df["triage_vital_o2"] > 100, "triage_vital_o2"] = np.nan

df["gender"] = df["gender"].astype(str).str.strip().str.lower().map(
    {"male": 0, "m": 0, "female": 1, "f": 1})

for col in VITALS + ["age", "gender"]:
    df[col] = df[col].fillna(df[col].median())

df["esi"] = df["esi"].astype(int)
print("Modelling table ready:", df.shape)
df["esi"].value_counts().sort_index()

Modelling table ready: (55121, 225)


,count
esi,
1,77
2,17924
3,27010
4,8896
5,1214


# Choose the features (X) and targets (y)

Here, we group each column by meaning. Vitals and CC flags here are primarily grouped. Columns like **leakage**, **admin/arriva**l and **demographics** are excluded. 

In [4]:
TARGET = "esi"

VITALS = ["triage_vital_hr", "triage_vital_sbp", "triage_vital_dbp", "triage_vital_rr",
          "triage_vital_o2", "triage_vital_temp", "triage_glucose"]

DEMOGRAPHICS = ["age", "gender", "ethnicity", "race", "lang", "religion",
                "maritalstatus", "employstatus", "insurance_status"]

ADMIN = ["dep_name", "arrivalmode", "arrivalmonth", "arrivalday", "arrivalhour_bin"]

LEAKAGE = ["disposition", "previousdispo"]

FEATURES = [c for c in df.columns if c != TARGET and c not in LEAKAGE + ADMIN + DEMOGRAPHICS]

X = df[FEATURES]
y = df[TARGET]
print("Model will use", len(FEATURES), "features. First few:", FEATURES[:6])

Model will use 208 features. First few: ['triage_vital_hr', 'triage_vital_sbp', 'triage_vital_dbp', 'triage_vital_rr', 'triage_vital_o2', 'triage_vital_o2_device']


In [5]:
#This is the exact Week 6 split 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print("train:", X_train.shape[0], "| test:", X_test.shape[0])

train: 44096 | test: 11025


# Discussion Point

Is *`cc_other`* worth keeping?

Since *`cc_other`* catches everything that the other flags didn't, it constitutes as a rare case that isn't usually considered. In my opinion, I'd say that the other flag be kept and investigated, rather than dismissed outright. After investigation, a decision can be made on whether the contents of the column are constructive and not just fluff.


In [6]:
if "cc_other" in df.columns:
    cc_cols = [c for c in df.columns if c.startswith("cc_")]
    total = len(df)
    has_other = int(df["cc_other"].sum())
    only_other = int(((df["cc_other"] == 1) & (df[cc_cols].sum(axis=1) == 1)).sum())
    print(f"Patients flagged cc_other: {has_other} of {total} ({has_other/total:.1%})")
    print(f"...and of those, patients whose ONLY complaint is 'other': {only_other}")
    print("\nMean ESI by cc_other flag (does 'other' lean urgent or not?):")
    print(df.groupby("cc_other")["esi"].mean().round(2))
else:
    print("No cc_other column in this sample — skipping. (On the full extract it will run.)")

Patients flagged cc_other: 4491 of 55121 (8.1%)
...and of those, patients whose ONLY complaint is 'other': 3352

Mean ESI by cc_other flag (does 'other' lean urgent or not?):
cc_other
0.0    2.87
1.0    3.01
2.0    3.26
3.0    3.00
Name: esi, dtype: float64


# Beating the Week 6 baseline

Here, the logistic regression baseline is retrained and the macro f1 is measured across all ESI levels. In this case, ESI 1 is treated equally as every other ESI level. From this point, every model has to beat this number to justify its extra cost. 

In [7]:
baseline = make_pipeline(StandardScaler(),
                         LogisticRegression(max_iter=1000, random_state=42))
baseline.fit(X_train, y_train)
baseline_f1 = f1_score(y_test, baseline.predict(X_test), average="macro")
print("Baseline (logistic regression) macro-F1:", round(baseline_f1, 3))

Baseline (logistic regression) macro-F1: 0.492


# Optimization

Generally, a model uses default settings that were chosen to be reasonable on average. This means that it isn't tuned to our data specifically; its unoptimized. Optimization means two things, feature engineering and hyperparameter tuning. 

**Feature engineering** is transforming raw data into meaningful, machine readable inputs. This gives the model better clues and info to go off of. **Hyperparameter tuning** is the process of selecting the optimal configuration of external settings. So, adjusting the model's own dials and settings. 

Both can lift performance without collecting a single new patient. 

# Feature Engineering

As mentioned before, a feature is a clue that the model uses. Feature engineering build new clues from existing columns using clinical knowledge. Two patterns, ratios/combinations and red flags are very handy in providing clues. We build them per row so that they're able to apply and train safely while also being able to be tests sparately, without leakage. 

NOTE: Blood pressure is a vital sign that is often mismeasured at triage. Other vitals, **respiratory rate**, **oxygen saturation**, **temperature** and **glucose**, can be used as features.

The code below builds new clinical features from existing vitals then applies them to the train and test splits in the same way.

In [8]:
def add_clinical_features(data):
    out = data.copy()

    # --- ratios & combinations (given as examples) ---
    out["shock_index"]    = out["triage_vital_hr"] / out["triage_vital_sbp"]       # HR / SBP         (uses BP)
    out["pulse_pressure"] = out["triage_vital_sbp"] - out["triage_vital_dbp"]      # SBP - DBP        (uses BP)
    out["spo2_rr_ratio"]  = out["triage_vital_o2"] / out["triage_vital_rr"]        # oxygen vs effort (NO BP)

    # --- added red flag combinatinns 
    out["is_tachypneic"] = (out["triage_vital_rr"]   > 20   ).astype(int)   # fast breathing
    out["is_hypoxic"]    = (out["triage_vital_o2"]   < 92   ).astype(int)   # low oxygen
    out["is_febrile"]    = (out["triage_vital_temp"] >= 100.4).astype(int)  # fever
    
    # --- a severity score = how many red flags fire:
    out["red_flag_count"] = out[["is_tachypneic", "is_hypoxic", "is_febrile"]].sum(axis=1)

    return out

X_train_fe = add_clinical_features(X_train)
X_test_fe = add_clinical_features(X_test)
print("Features after engineering:", X_train_fe.shape[1])
X_train_fe.head()

Features after engineering: 215


,triage_vital_hr,triage_vital_sbp,triage_vital_dbp,triage_vital_rr,triage_vital_o2,triage_vital_o2_device,triage_vital_temp,triage_glucose,cc_abdominalcramping,cc_abdominaldistention,...,cc_woundre-evaluation,cc_wristinjury,cc_wristpain,shock_index,pulse_pressure,spo2_rr_ratio,is_tachypneic,is_hypoxic,is_febrile,red_flag_count
35369,104.0,120.0,71.0,22.0,98.0,1.0,98.2,137.0,0.0,0.0,...,0.0,0.0,0.0,0.866667,49.0,4.454545,1,0,0,1
52043,78.0,115.0,76.0,18.0,96.0,0.0,98.4,102.0,0.0,0.0,...,0.0,0.0,0.0,0.678261,39.0,5.333333,0,0,0,0
13610,96.0,119.0,78.0,18.0,94.0,0.0,98.1,108.0,0.0,0.0,...,0.0,0.0,0.0,0.806723,41.0,5.222222,0,0,0,0
54796,89.0,128.0,93.0,16.0,98.0,0.0,97.7,108.0,0.0,0.0,...,0.0,0.0,0.0,0.695312,35.0,6.125000,0,0,0,0
11096,89.0,113.0,78.0,18.0,98.0,0.0,98.1,92.0,0.0,0.0,...,0.0,0.0,0.0,0.787611,35.0,5.444444,0,0,0,0


# M1 - Random Forest

Random forest is an algorithm that creates multiple decision trees and merges them together to get a more accurate and stable prediction. It operates using the "wisdom of crows" principle, meaning that a group of independent models makes better, more robust decisions than any single model on its own. One tree can overfit, but, a whole forst averages out the noise. Random forest requires no scaling and hands you feature importances, how much each clue mattered. Thus, it takes each ESI level into consideration instead of skewing the data towards the most available. 

It works by having each tree trained on a random subset of the training data, having each tree see slightly different data. This is called **bootstrapping**. Similarly, instead of having each tree look at every feature to make a decision, it randomly selects a smaller subset of features at each step. This is called **feature randomness**. 

The code below shows the implementation of the random forest algorithm based on the feature engineering done beforehand, as well as the targets. The macro f1 score for each ratio/combination and red flag is also computed and shown.

In [9]:
rf = RandomForestClassifier(n_estimators = 300, class_weight = "balanced", random_state = 42, n_jobs = -1)
rf.fit(X_train_fe, y_train)
rf_f1 = f1_score(y_test, rf.predict(X_test_fe), average = "macro")
rf_rank = pd.Series(rf.feature_importances_, index = X_train_fe.columns)


print(rf_f1)
print(rf_rank)

0.3901272390518964
triage_vital_hr     0.067139
triage_vital_sbp    0.077729
triage_vital_dbp    0.072644
triage_vital_rr     0.027916
triage_vital_o2     0.038061
                      ...   
spo2_rr_ratio       0.049601
is_tachypneic       0.005882
is_hypoxic          0.002229
is_febrile          0.000978
red_flag_count      0.007575
Length: 215, dtype: float64


# Encoding Categorical Features

Machine learning models only understand numbers and code. As a result, text labelled columns such as gender, ethnicity and race must be converted to numbers first. They can either be: 
* Converted using binary, like where gender becomes 0/1 for M/F
* Have multi-category columns become one-hot

Earlier demographics were left out of the base model, the code below shows how the column would be reintroduced. As mentioned before, it works by turining text demographic into numbers:
* **gender**                ->      already has binary from the cleaning
* **age**                  ->      already numeric
* **ethicity** and **race**     ->      uses one-hot to bring a new binary column per category

In [10]:
demo_1hot = pd.get_dummies(df[["ethnicity", "race"]], prefix = ["eth", "race"], dtype = int)
print("New one-hot columns:", list(demo_1hot.columns))

def add_demographics(X_fe):
    """Bolt the encoded demographics onto an existing feature frame (aligned by row)."""
    rows = X_fe.index
    extra = demo_1hot.loc[rows].copy()
    extra["age"] = df.loc[rows, "age"]         # numeric already
    extra["gender"] = df.loc[rows, "gender"]   # 0/1 already
    return pd.concat([X_fe, extra], axis=1)

X_train_plus = add_demographics(X_train_fe)
X_test_plus = add_demographics(X_test_fe)
print("Features WITHOUT demographics:", X_train_fe.shape[1])
print("Features WITH    demographics:", X_train_plus.shape[1])
X_train_plus.filter(like="race_").head()

New one-hot columns: ['eth_Hispanic or Latino', 'eth_Non-Hispanic', 'eth_Patient Refused', 'eth_Unknown', 'race_American Indian or Alaska Native', 'race_Asian', 'race_Black or African American', 'race_Native Hawaiian or Other Pacific Islander', 'race_Other', 'race_Patient Refused', 'race_Unknown', 'race_White or Caucasian']
Features WITHOUT demographics: 215
Features WITH    demographics: 229


,race_American Indian or Alaska Native,race_Asian,race_Black or African American,race_Native Hawaiian or Other Pacific Islander,race_Other,race_Patient Refused,race_Unknown,race_White or Caucasian
35369,0,0,0,0,1,0,0,0
52043,0,0,0,0,0,0,0,1
13610,0,0,0,0,0,0,0,1
54796,0,0,1,0,0,0,0,0
11096,0,0,0,0,0,0,0,1


## Discussion
*Which demographics actually earn a place in the model?*

1. Clinically justified and ethically fine:
* `age`, `gender`

2. Fairness-sensitive:
* `race`, `ethnicity`, `religion`

3. Fluff:
* `maritalstatus`, `employmentstatus`, `insurancestatus`

In my opinion, only the clinically justified, and less so fairness-sensitive, earn their place in the model. While `age` and `gender` are must haves and are actual indicators of issues in the medical field, other demographics, like the ones under fairness-sensitive, can give the model something to learn from. For example, sickle cell primarily affects people of Black and African decent. Rarely does it affect other races/ethnicities. Similarly, different religions practice their own forms of healing, such as herbs, medicines, physical practices, etc. that keep them healthy, reduce pain or even cure them from a disease outright. This could explain why individuals who are affiliated with a certain type of religion, or are religious overall, are less/more likely to visit the ED, stay in the ED for less/more amounts of time, etc.  

## Note
Using race and ethnicity as model inputs is ethically sensitive. A model that leans on them can entrench bias and a higher accuracy score is not enough to justify it. Understand their effect on the model, but keep those inputs out of the model unless the choice and governance can be defended and signed off. 

In [11]:
rf_demo = RandomForestClassifier(n_estimators = 300, class_weight = "balanced", random_state = 42, n_jobs = -1)
rf_demo.fit(X_train_plus, y_train)
rf_demo_f1 = f1_score(y_test, rf_demo.predict(X_test_plus), average = "macro")

print(f"Demo F1: {rf_demo_f1}")
print(f"Real F1: {rf_f1}")


Demo F1: 0.3893232958084241
Real F1: 0.3901272390518964


## Observations
Noting these results, seeing that the real f1 = 0.3901 and the demo f1 = 0.3893, it seems as if the difference is negligible. Adding the one-hot demographic data, or just simply leaving it out, doesn't truly make a difference for this dataset. However, this result doesn't tell what would happen for another dataset. This simply means that the demographic data in this dataset isn't contributory to the actual important information that the model looks at, which is primarily the vitals, ESI and CC flags. 

Additionally, the model isn't set up to have these demographics carry significant weight. For this model, we're looking at the important features, like the ones mentioned previously, the focus isn't on demographics and how they may or may not affect triage. The root of the model lies within this question, "How effectively can it triage a patient?". Once this question can be answered close to flawlessly by the model, the demographic data isn't required at all. Hence why the observations carry negligible returns. 

# Hyperparameter Tuning with Cross-Validation
As mentioned before, *hyperparameters* are the model's own dials. For a forest, its how many trees, how deep it goes, how many samples per leaf, etc.  

Tuning hyperparameters by repeatedly checking performance on the same test set is running the risk of leaking knowledge of the test set into the choices. As a result, the evaluation is no longer a true generalization. Here, *cross-validation* solves the issue without needing to separate the set. Cross-validation is used to split the training data into folds and rotates which fold is held out. As a result, a setting has to do well on several other mini-tests, rather than getting lucky only once.

**GridSearchCV** - Tries every combo | Slow
Use this when the dataset is small and the output should be a guaranteed extensive answer

**RandomizedSearchCV** - Samples a number of random combos | Fast
Use this when the space is large, as it can find good parameters for a fraction of the compute cost as GridSearchCV

The code below shows RandomizedSearchCV being ran over a few random forst settings

In [ ]:
param_dist = {
    "n_estimators": [100, 200, 300, 400],
    "max_depth": [None, 6, 10, 16],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": ["sqrt", "log2", None],
}

search = RandomizedSearchCV(RandomForestClassifier(class_weight = "balanced", random_state = 42), param_distributions = param_dist, n_iter = 8, cv = 8, scoring = "f1_macro")

search.fit(X_train_fe, y_train)

print(search.best_params_)
print(search.best_score_)
print(search.best_estimator_)

# M2 - Gradient Boosting

Gradient boosting is a technique that builds a strong predictive model by sequentially combining multiple weak models, like decision trees, and gradually trains them one after the other. Intead of training each model independently, gradient boosting ensures that each one corrects the errors made by the preceeding one. This method iteratively improves accuracy at the expense of computing power. On tabular data, like this one, it's the strongest model. 

**XGBoost** and **LightBGM** are popular external libraries that do gradient boosting. 

In [ ]:
hgb = HistGradientBoostingClassifier(
    max_depth=6, learning_rate=0.1, max_iter=300,
    class_weight="balanced", random_state=42)
hgb.fit(X_train_fe, y_train)
hgb_f1 = f1_score(y_test, hgb.predict(X_test_fe), average="macro")
print("Gradient Boosting macro-F1:", round(hgb_f1, 3))

# M3 - Small Neural Network (MLP)
A multi-layert perceptron (MLP) is a small neural network. On tabular data it rearely beats good tree models and is harder to explain. It needs scaling, like logistic regression, so we can wrap it in a pipeline with `StandardScaler`. 

The code below shows the implementation of a small neural network on scaled features

In [ ]:
mlp = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(64, 32), alpha=1e-3, max_iter=500, random_state=42))
mlp.fit(X_train_fe, y_train)
mlp_f1 = f1_score(y_test, mlp.predict(X_test_fe), average="macro")
print("Small MLP macro-F1:", round(mlp_f1, 3))

# Hyperparameter Cheat Sheet

## Random Forest
* `n_estimators`              -> How many trees are in the forest. More trees = steadier predictions, but slower
* `max_depth`                 -> How many yes/no questions deep can each tree go. Deeper fits more detail, but risks memorising noise (`None` = grow until pure)
* `min_sanmples_leaf`         -> The fewest patients allowed in a final leaf. Bigger = smoother, less overfitting
* `max_features`              -> How many features each split may consider. Fewer = more variety between trees
* `class_weight = "balanced"` -> Tells it to pay extra attention to rare classes, like ESI 1

## Gradient Boosting
* `learning_rate`     -> How big a correction each new tree makes. Smaller = more careful, usually needs more rounds
* `max_iter`          -> How many boosting rounds (trees built one after another)
* `max_depth`         -> Depth of each individual tree. Boosting usually uses shallow trees
* `class_weight`      -> As above, "balanced" up-weights the rare, critical classes

## Small Neural Network (MLP)
* `hidden_layer_sizes`    -> the shape of the network, e.g. (64, 32) = two hidden layers of 64 then 32 neurons. Bigger = more capacity, more overfit risk, more compute.
* `alpha`                 -> regularisation strength: higher = a simpler model that overfits less.
* `max_iter`              -> how many training passes before it stops.


# Scoreboard

In [ ]:
scores = {
    "Baseline (LogReg)": baseline_f1,
    "Random Forest": rf_f1,
    "Random Forest (tuned)": rf_tuned_f1,
    "Gradient Boosting": hgb_f1,
    "Small MLP": mlp_f1,
}
pd.Series(scores).sort_values(ascending=False).round(3).to_frame("macro_F1")

# Saved Model

In [ ]:
joblib.dump(rf_tuned, "w7_random_forest.joblib")
joblib.dump(hgb, "w7_gradient_boosting.joblib")
joblib.dump(mlp, "w7_mlp.joblib")
print("Saved: w7_random_forest.joblib, w7_gradient_boosting.joblib, w7_mlp.joblib")

#Reload later
#rf_tuned = joblib.load("w7_random_forest.joblib")